# Optimization and Curve Fitting 🎯

Optimization algorithms are essential mathematical/computational tools widely utilized all across scientific disciplines.
A few examples:
- Fitting experimental data to physical laws or theories  
- Determine the lowest-energy conformations of drug molecules
- Construction of a machine learning model ($f: X \rightarrow Y$)
- Metabolic network modeling<br>

There are many types of optimization algorithms, ranging from finding local maxima to finding global minima. 
  
Local optimization algorithms find function extrema within a neighborhood surrounding an initial starting point. These minima or maxima are of interest to study (e.g. what point has the smallest error between raw data and a physical law). We will focus on finding local minima.  
- Local minimum is the lowest point in a "neighborhood"
- Where as the global minimum is the function's lowest point across it's entire defined domain
  
  $\frac{\partial f}{\partial x_{i}} = 0;$ $\frac{\partial^{2} f}{\partial x^{2}_{i}} > 0$

Even though we wil not go into global optimization, I wanted to note that a number of the global optimization algorithms were inspired by biology.
- Genetic Algorithms (GAs)
- Differential Evolution (DE)
- Particle Swarm Optimization (PSO)
---

## **Energy Minimzation / Geometry Optimization : Finding low energy conformations**

An important concept for understanding geometry optimization is the potential energy surface (PES) or landscape. The PES describes of the energy system along each degree of freedom (DOF). DOF represents independent parameters that are free to vary.

---

In [ ]:
import numpy as np
import plotly.graph_objects as go

# 1. Define your specific function
def f(x, y):
    return np.sin(x) * np.cos(y)

# 2. Build the coordinate matrix grid
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
Z = f(X, Y)
    
# 3. Generate the 3D plot
fig = go.Figure(data=[go.Surface(z=Z, x=x, y=y, colorscale='Viridis')])
fig.update_layout(
    title='Potential Energy Surface',
    autosize=False,
    width=500,
    height=500,
    scene=dict(
        xaxis=dict(title='Coordinate 1'),
        yaxis=dict(title='Coordinate 2'),
        zaxis=dict(title='Potential Energy')
    ),
    margin=dict(l=65, r=50, b=65, t=90)
)
fig.show()



How would you identify a local minimum starting from an arbitrary point on this surface? 

---



Gradient descent:

$\mathbf{x_{n+1}} = \mathbf{x_{n}} - \eta \nabla f(\mathbf{x_{n}})$

where $\eta$ is the step size or learning rate.<br><br>

For 3D caresian coordinate system,<br><br>
$\nabla f(\mathbf{x_{n}}) = \frac{\partial f}{\partial x}\mathbf{i} + \frac{\partial f}{\partial y}\mathbf{j} + \frac{\partial f}{\partial z}\mathbf{k}$

where **i**,**j**,**k** are the standard unit vectors in the direction of x,y,z 


---

In [ ]:
import numpy as np
import plotly.graph_objects as go

# 1. Define the function and its gradient
def f(x, y):
    return np.sin(x) * np.cos(y)

def grad_f(x, y):
    # Partial derivatives for gradient descent
    df_dx = np.cos(x) * np.cos(y)
    df_dy = -np.sin(x) * np.sin(y)
    return np.array([df_dx, df_dy])

# 2. Generate the background PES grid
x_grid = np.linspace(-3.5, 3.5, 150)
y_grid = np.linspace(-3.5, 3.5, 150)
X, Y = np.meshgrid(x_grid, y_grid)
Z = f(X, Y)

# 3. Simulate Optimization Path (Gradient Descent)
current_pos = np.array([1.48, 0.2])  # Starting coordinates
# current_pos = np.array([1.48, 0.0])  # Starting coordinates
learning_rate = 0.2
iterations = 50

path_x = [current_pos[0]]
path_y = [current_pos[1]]
path_z = [f(current_pos[0], current_pos[1])]

for _ in range(iterations):
    grad = grad_f(current_pos[0], current_pos[1])
    current_pos = current_pos - learning_rate * grad  # Step towards minimum
    path_x.append(current_pos[0])
    path_y.append(current_pos[1])
    path_z.append(f(current_pos[0], current_pos[1]))


fig = go.Figure()
# Add static PES surface
fig.add_trace(go.Surface(
    z=Z, x=x_grid, y=y_grid, 
    colorscale='Viridis', 
    opacity=0.8,
    showscale=False
))

# Add the moving red dot (initialized at step 0)
fig.add_trace(go.Scatter3d(
    x=[path_x[0]], y=[path_y[0]], z=[path_z[0]],
    mode='markers',
    marker=dict(color='red', size=8, symbol='circle'),
    name='Optimizer'
))

# 5. Create Animation Frames
frames = []
for i in range(len(path_x)):
    frames.append(go.Frame(
        data=[
            go.Surface(),  # Keep surface static by passing empty placeholder
            go.Scatter3d(x=[path_x[i]], y=[path_y[i]], z=[path_z[i]])  # Move the dot
        ],
        name=f'step{i}'
    ))
fig.frames = frames

# 6. Add Animation Controls & Layout
fig.update_layout(
    title='Optimization Path on PES',
    autosize=False,
    width=500,
    height=500,
    scene=dict(
        xaxis=dict(title='Coordinate 1', range=[-3.5, 3.5]),
        yaxis=dict(title='Coordinate 2', range=[-3.5, 3.5]),
        zaxis=dict(title='Potential Energy', range=[-1.1, 1.1])
    ),
    updatemenus=[dict(
        type='buttons',
        showactive=False,
        y=0,
        x=1.05,
        xanchor='right',
        yanchor='top',
        buttons=[
            dict(
                label='▶ Play',
                method='animate',
                args=[None, dict(frame=dict(duration=100, redraw=True), fromcurrent=True)]
            ),
            dict(
                label='⏸ Pause',
                method='animate',
                args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')]
            )
        ]
    )]
)

fig.show()

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from ase import Atoms
from ase.optimize import BFGS
from ase.io.trajectory import Trajectory
from fairchem.core import FAIRChemCalculator, pretrained_mlip

import nglview as nv


# 1. Build the 3D Peptide using RDKit
peptide_smiles = "C[C@@H](C(=O)N[C@@H](C)C(=O)N[C@@H](C)C(=O)O)N"  
mol = Chem.MolFromSmiles(peptide_smiles)
mol_with_hs = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol_with_hs) 

# 2. Convert RDKit Molecule to ASE Atoms object
positions = mol_with_hs.GetConformer().GetPositions()
symbols = [atom.GetSymbol() for atom in mol_with_hs.GetAtoms()]
ase_atoms = Atoms(symbols=symbols, positions=positions)

# 3. Setup and Run Geometry Optimization with UMA ML Force Field
predictor = pretrained_mlip.get_predict_unit("uma-s-1p1")
calculator = FAIRChemCalculator(predictor, task_name="omol")
ase_atoms.calc = calculator
traj = Trajectory("peptide_optimization.traj", "w", ase_atoms)
dyn = BFGS(ase_atoms, trajectory=traj)
dyn.run(fmax=0.05)  # Optimizes until residual forces are small
traj.close()        # Ensure file buffer flushes completely

In [ ]:
optimization_history = Trajectory("peptide_optimization.traj")
view = nv.show_asetraj(optimization_history)
# (Optional) Enhance the rendering style
view.add_representation("ball+stick", selection="all")
view

## **Curve Fitting / Optimization**


Curve fitting is typically a computational algorithm that generates the "best" fit of a set of data points to a mathematical function.  
 
- Physical Sciences (fit to a function that describes a physical law) 
- Machine learning (Tries to learn patterns/relationships in a data set)
  - Physics-Informed Machine Learning 
  
An important concept to understand curve fitting is the loss function. The loss function describes the performance of the fit/model by calculating deviation of the model’s predictions from the correct, “ground truth” predictions. 
Examples:
- Mean squared error: $MSE = \frac{1}{n} \sum_{n=1}^{n}(Y_{i} - Y^{ref}_{i})^2$ 
- Mean absolute error: $MAE = \frac{1}{n} \sum_{n=1}^{n}|Y_{i} - Y^{ref}_{i}|$
          
---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def func(x, m, b):
    return m * x + b

def generate_example_data():
    xdata = np.linspace(0, 4, 50)
    y = func(xdata, 2.5, 1.3)
    ## change slope and intercept for the first 25 points
    y[:25] = func(xdata[:25], 2.0, 1.0)
    # add some noise to the data
    y_noise = np.random.normal(scale=0.8, size=xdata.size)
    ydata = y + y_noise
    return xdata, ydata

xdata, ydata = generate_example_data()
plt.plot(xdata, ydata, 'o', color='black', label='data')

popt, _ = curve_fit(func, xdata, ydata)
plt.plot(xdata, func(xdata, *popt), 'r-',
         label='fit: m=%5.3f, b=%5.3f' % tuple(popt))
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()